# First-pass liquidity forecast & rebalancing rule (squad draft)

This is the squad's first attempt at forecasting account balances and recommending
daily transfers between accounts to avoid shortfalls. Leadership isn't confident in
it and a rollout decision is pending — the forecasts look noisy and the recommended
transfers don't always make sense.

**Data:** `accounts.csv`, `account_balances_daily_RAW.csv`, `transfers_log_RAW.csv`


In [ ]:
import pandas as pd
import numpy as np

accounts = pd.read_csv("accounts.csv")
balances = pd.read_csv("account_balances_daily_RAW.csv")
transfers = pd.read_csv("transfers_log_RAW.csv")

balances["date"] = pd.to_datetime(balances["date"], errors="coerce")  # a few dates don't parse, we just drop those with dropna() below
balances = balances.dropna(subset=["date"])
balances = balances.sort_values(["account_id", "date"])
balances.head()


,date,account_id,currency,balance,inflow,outflow
1251,2025-01-01,ACC-001,COP,2.579044e+09,9.404482e+07,1.500099e+07
1473,2025-01-02,ACC-001,COP,2.561628e+09,1.257575e+08,1.431733e+08
895,2025-01-03,ACC-001,COP,2.561628e+09,0.000000e+00,0.000000e+00
144,2025-01-05,ACC-001,COP,2.617987e+09,2.996917e+07,0.000000e+00
1387,2025-01-06,ACC-001,COP,2.619401e+09,1.360959e+08,1.346818e+08


## Step 1 — Quick look at the data

We didn't see anything obviously broken, so we moved on to forecasting.


In [ ]:
print(balances.shape)
print(balances["currency"].unique())
balances.describe()


(1375, 6)
['COP' 'Colombian Peso' 'cop' 'USD' 'US Dollar' 'usd' 'MXN' 'mxn'
 'Mexican Peso']


,date,balance,inflow,outflow
count,1375,1.316000e+03,1.320000e+03,1.328000e+03
mean,2025-05-15 01:36:20.945454336,1.012365e+09,2.644444e+07,2.803600e+07
min,2025-01-01 00:00:00,-4.070746e+09,0.000000e+00,0.000000e+00
25%,2025-03-08 00:00:00,1.754911e+05,3.856918e+03,4.216882e+03
50%,2025-05-13 00:00:00,1.037867e+08,1.205203e+06,1.427930e+06
75%,2025-07-23 00:00:00,2.536516e+09,3.878938e+07,3.073473e+07
max,2025-09-27 00:00:00,4.094989e+09,2.393587e+08,5.557494e+08
std,NaN,1.434256e+09,4.628130e+07,5.189343e+07


## Step 2 — Forecast next 7 days per account

Simple approach: use the trailing 14-day average of balance as the forecast for the
next week. Fast to implement, easy to explain to stakeholders.


In [ ]:
def forecast_next_week(df, account_id, window=14):
    sub = df[df.account_id == account_id].sort_values("date")
    trailing = sub["balance"].tail(window).mean()
    return trailing

forecasts = {}
for acc in accounts.account_id:
    forecasts[acc] = forecast_next_week(balances, acc)

forecasts


{'ACC-001': np.float64(2305518572.105),
 'ACC-002': np.float64(79789.68115310009),
 'ACC-003': np.float64(2920535273.396059),
 'ACC-004': np.float64(15965981.78142842),
 'ACC-005': np.float64(107312810.28428571),
 'ACC-006': np.float64(151185.90153846153)}

## Step 3 — Rebalancing rule

If an account's forecast is below a fixed threshold, pull funds from whichever
account currently has the highest balance.


In [ ]:
THRESHOLD = {
    "ACC-001": 1_000_000_000,
    "ACC-002": 100_000,
    "ACC-004": 40_000_000,
    "ACC-006": 100_000,
}

latest_balance = balances.sort_values("date").groupby("account_id")["balance"].last()

recommendations = []
for acc, threshold in THRESHOLD.items():
    if forecasts[acc] < threshold:
        # pick the account with the single highest raw balance across ALL currencies
        donor = latest_balance.drop(index=acc).idxmax()
        shortfall = threshold - forecasts[acc]
        recommendations.append({
            "to_account": acc,
            "from_account": donor,
            "amount_needed": round(shortfall, 2),
            "transfer_today": True,
        })

recommendations


[{'to_account': 'ACC-002',
  'from_account': 'ACC-003',
  'amount_needed': np.float64(20210.32),
  'transfer_today': True},
 {'to_account': 'ACC-004',
  'from_account': 'ACC-003',
  'amount_needed': np.float64(24034018.22),
  'transfer_today': True}]

## Step 4 — Did the last round of transfers help?

We compare average balances before vs. after we started actively rebalancing
(May 1st) to see if the shortfalls got less frequent.


In [ ]:
cutover = pd.Timestamp("2025-05-01")
pre = balances[balances.date < cutover]["balance"].mean()
post = balances[balances.date >= cutover]["balance"].mean()

print(f"Avg balance before rebalancing: {pre:,.0f}")
print(f"Avg balance after rebalancing:  {post:,.0f}")
print(f"Looks like it's working, balances are up {100*(post/pre - 1):.1f}%")


Avg balance before rebalancing: 1,068,097,439
Avg balance after rebalancing:  965,805,394
Looks like it's working, balances are up -9.6%


## Conclusion (squad's draft)

- The 14-day trailing average is a reasonable, simple forecast.
- The rebalancing rule successfully identifies which account has spare cash.
- Balances are up ~X% since we started rebalancing on May 1st, so the approach
  seems to be working. We'd like sign-off to roll this out to all accounts.

**Open question we couldn't resolve:** the forecast for ACC-004 sometimes comes
out negative or wildly different day to day, and the donor account picked by the
rule doesn't always make sense (once it picked a MXN reserve account to cover a
COP shortfall). We think it's just noise in the data and were about to add more
smoothing when we got pulled onto another priority.
